<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def kv_cache_size(
    num_layers,
    hidden_size,
    sequence_length,
    batch_size=1,
    bytes_per_element=2,
):
    """
    Calculate KV cache memory.

    Args:
        num_layers (int): Number of transformer layers
        hidden_size (int): Hidden dimension (d_model)
        sequence_length (int): Total cached tokens
        batch_size (int): Number of sequences
        bytes_per_element (int): 2 for FP16/BF16, 4 for FP32

    Returns:
        Tuple (bytes, MB, GB)
    """

    kv_bytes = (
        2
        * num_layers
        * batch_size
        * sequence_length
        * hidden_size
        * bytes_per_element
    )

    mb = kv_bytes / (1024 ** 2)
    gb = kv_bytes / (1024 ** 3)
    return kv_bytes, mb, gb

bytes_used, mb, gb = kv_cache_size(
    32,
    hidden_size=4096,
    sequence_length=2048,
    batch_size=1,
    bytes_per_element=2,
)


print(f"KV Cache:")
print(f"Bytes: {bytes_used:,}")
print(f"MB    : {mb:.2f}")
print(f"GB    : {gb:.2f}")

KV Cache:
Bytes: 1,073,741,824
MB    : 1024.00
GB    : 1.00


In [ ]:
from enum import Enum
from dataclasses import dataclass, field
from typing import List
import time


class Status(Enum):
  WAITING = "waiting"
  RUNNING = "running"
  FINISHED = "finished"

@dataclass

class Request:
  req_id:  str
  prompt_tokens: List[int]
  max_new_tokens: int

  status: Status = Status.WAITING
  generated_tokens: List[int] = field(default_factory=list)
  arrival_time: float = field(default_factory=time.time)
  start_time: float = None
  finish_time: float = None

  def is_finished(self):
    return len(self.generated_tokens) >= self.max_new_tokens

  def num_tokens(self):
    return len(self.prompt_tokens) + len(self.generated_tokens)

  def ttft(self):
    if self.start_time is None:
      return None
    return self.start_time - self.arrival_time

In [ ]:
req = Request("req_0", prompt_tokens=[1,2,3,4], max_new_tokens=5)
print(req.status)
print(req.num_tokens())
print(req.is_finished())

Status.WAITING
4
False


In [ ]:
@dataclass
class Block:
  block_id: int
  tokens_stored: int = 0
  is_free: bool = True

  BLOCK_SIZE = 4

class KVCacheManager:

  def __init__(self, total_blocks: int):
    self.total_blocks = total_blocks
    self.all_blocks = [Block(i) for i in range(total_blocks)]
    self.free_pool = list(self.all_blocks)
    self.block_table = {}

  def free_block_count(self):
    return len(self.free_pool)

  def can_allocate(self, req: Request) -> bool:

    import math
    needed = math.ceil(len(req.prompt_tokens) / Block.BLOCK_SIZE)
    return len(self.free_pool) >= needed

  def allocated(self, req:Request):
    import math
    needed = math.ceil(len(req.prompt_tokens)/ Block.BLOCK_SIZE)
    blocks = []

    for _ in range(needed):
      block = self.free_pool.pop()
      block.is_free = False
      block.tokens_stored = 0
      blocks.append(block)
    self.block_table[req.req_id] = blocks

    remaining = len(req.prompt_tokens)
    for block in blocks:
      block.tokens_stored = min(Block.BLOCK_SIZE, remaining)
      remaining -= block.tokens_stored

  def append_token(self, req:Request) -> bool:
    blocks = self.block_table[req.req_id]
    last_block = blocks[-1]

    if last_block.tokens_stored < Block.BLOCK_SIZE:
      last_block.tokens_stored +=1
      return False
    else:
      if not self.free_pool:
        return None
      new_block = self.free_pool.pop(0)
      new_block.is_free = False
      new_block.tokens_stored = 1
      blocks.append(new_block)
      return True

  def free(self, req: Request):
    if req.req_id not in self.block_table:
      return
    for block in self.block_table[req.req_id]:
      block.tokens_stored = 0
      block.is_free = True
      self.free_pool.append(block)
    del self.block_table[req.req_id]

  def used_blocks(self, req:Request) -> int:
    return len(self.block_table.get(req.req_id, []))

In [ ]:
kv = KVCacheManager(total_blocks=10)
req = Request("req_0", prompt_tokens=[1,2,3,4,5], max_new_tokens=10)

print(kv.can_allocate(req))
kv.allocated(req)
print(kv.free_block_count())
kv.free(req)
print(kv.free_block_count())

True
8
10


The Engine - for inference

In [ ]:
import random

class Engine:

  def step(self, batch: List[Request]) -> dict:

    outputs = {}
    for req in batch:
      new_token = random.randint(1,1000)
      outputs[req.req_id] = new_token
    return outputs

In [ ]:
from collections import deque

class Scheduler:
  def __init__(self, total_blocks: int=20,
               max_batch_size: int = 4):
    self.kv_manager = KVCacheManager(total_blocks)
    self.engine = Engine()
    self.max_batch_size = max_batch_size

    self.waiting = deque()
    self.running = []
    self.finished = []

  def add_request(self, req: Request):
    self.waiting.append(req)

  def _admit(self):

    while self.waiting:
      if len(self.running) >= self.max_batch_size:
        break
      req = self.waiting[0]

      if not self.kv_manager.can_allocate(req):
              break

      self.waiting.popleft()
      self.kv_manager.allocated(req)
      req.status = Status.RUNNING
      req.start_time = time.time()
      self.running.append(req)

  def _step(self):
    if not self.running:
      return
    outputs = self.engine.step(self.running)

    for req in self.running:
      token = outputs[req.req_id]
      req.generated_tokens.append(token)

      self.kv_manager.append_token(req)

  def _finish(self):
    still_running = []
    for req in self.running:
      if req.is_finished():
        req.status = Status.FINISHED
        req.finish_time = time.time()
        self.kv_manager.free(req)
        self.finished.append(req)
      else:
        still_running.append(req)
    self.running = still_running


  def _print_state(self, step: int):
        running_str = " ".join(
            f"{r.req_id}({len(r.generated_tokens)}tok)"
            for r in self.running
        )
        waiting_str = " ".join(r.req_id for r in self.waiting)
        finished_str = " ".join(r.req_id for r in self.finished)
        free = self.kv_manager.free_block_count()
        total = self.kv_manager.total_blocks

        print(f"step {step:02d} | "
              f"running: [{running_str}] | "
              f"waiting: [{waiting_str}] | "
              f"finished: [{finished_str}] | "
              f"free blocks: {free}/{total}")

  def run(self):
    step = 0
    while self.waiting or self.running:
      self._admit()
      self._step()
      self._finish()
      self._print_state(step)
      step+=1

    self._print_stats()


  def _print_stats(self):
        print("\n stats ")
        for req in self.finished:
            print(f"{req.req_id} | "
                  f"ttft: {req.ttft():.3f}s | "
                  f"total tokens: {req.num_tokens()} | "
                  f"prompt: {len(req.prompt_tokens)} | "
                  f"generated: {len(req.generated_tokens)}")

In [ ]:
if __name__ == "__main__":

    scheduler = Scheduler(total_blocks=20, max_batch_size=4)

    requests = [
        Request("req_0", prompt_tokens=list(range(5)),  max_new_tokens=8),
        Request("req_1", prompt_tokens=list(range(3)),  max_new_tokens=5),
        Request("req_2", prompt_tokens=list(range(7)),  max_new_tokens=10),
        Request("req_3", prompt_tokens=list(range(2)),  max_new_tokens=6),
        Request("req_4", prompt_tokens=list(range(4)),  max_new_tokens=4),
    ]

    for req in requests:
        scheduler.add_request(req)

    scheduler.run()

step 00 | running: [req_0(1tok) req_1(1tok) req_2(1tok) req_3(1tok)] | waiting: [req_4] | finished: [] | free blocks: 14/20
step 01 | running: [req_0(2tok) req_1(2tok) req_2(2tok) req_3(2tok)] | waiting: [req_4] | finished: [] | free blocks: 12/20
step 02 | running: [req_0(3tok) req_1(3tok) req_2(3tok) req_3(3tok)] | waiting: [req_4] | finished: [] | free blocks: 11/20
step 03 | running: [req_0(4tok) req_1(4tok) req_2(4tok) req_3(4tok)] | waiting: [req_4] | finished: [] | free blocks: 10/20
step 04 | running: [req_0(5tok) req_2(5tok) req_3(5tok)] | waiting: [req_4] | finished: [req_1] | free blocks: 12/20
step 05 | running: [req_0(6tok) req_2(6tok) req_4(1tok)] | waiting: [] | finished: [req_1 req_3] | free blocks: 11/20
step 06 | running: [req_0(7tok) req_2(7tok) req_4(2tok)] | waiting: [] | finished: [req_1 req_3] | free blocks: 11/20
step 07 | running: [req_2(8tok) req_4(3tok)] | waiting: [] | finished: [req_1 req_3 req_0] | free blocks: 14/20
step 08 | running: [req_2(9tok)] | wait